У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTENC
from imblearn.combine import SMOTETomek
from sklearn.multiclass import OneVsRestClassifier

np.set_printoptions(legacy='1.25')
pd.set_option('display.max.columns', 100)

In [3]:
raw_df = pd.read_csv("drive/MyDrive/Colab Notebooks/data/customer_segmentation_train.csv")

In [4]:
raw_df.head(5)

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [5]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


In [6]:
raw_df.isna().sum() / len(raw_df) * 100

,0
ID,0.000000
Gender,0.000000
Ever_Married,1.735250
Age,0.000000
Graduated,0.966782
Profession,1.536936
Work_Experience,10.275161
Spending_Score,0.000000
Family_Size,4.152206
Var_1,0.941993


In [7]:
raw_df.Spending_Score.unique()

array(['Low', 'Average', 'High'], dtype=object)

In [8]:
# Create inputs and targets
input_cols = list(raw_df.columns)[1:-1]
target_col = 'Segmentation'

train_inputs, test_inputs, train_targets, test_targets = train_test_split(
    raw_df[input_cols], raw_df[target_col],
    test_size=0.2,
    random_state=42,
    stratify=raw_df[target_col]
)

# Identify numeric and categorical columns
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
ordinal_cols = ['Spending_Score']
categorical_cols = (train_inputs.drop(ordinal_cols, axis=1).select_dtypes('object').columns.to_list())

# Create preprocessing pipelines for both numeric and categorical data
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy = 'mean')),
    ('scaler', MinMaxScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ord_enc', OrdinalEncoder(categories=[['Low', 'Average', 'High']]))
])

# Combine transformers for different types of columns into one preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
        ('ord', ordinal_transformer, ordinal_cols)
    ])

# Create a pipeline that includes preprocessing and the model
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='lbfgs', max_iter=1000))
])

# Train the model
model_pipeline.fit(train_inputs, train_targets)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Age', 'Work_Experience',
                                                   'Family_Size']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Gender', 'Ever_Married',
                                                   'Graduated', 'Profession',
                                                   'Var_1']),
                                                 ('ord',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ord_enc',
                                                                   OrdinalEncoder(categories=[['Low',
                                                                                               'Average',
                                                                                               'High']]))]),
                                                  ['Spending_Score'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [9]:
# Predict on test set
test_preds = model_pipeline.predict(test_inputs)

# Evaluate
print("Classification Report:\n", classification_report(test_targets, test_preds))

Classification Report:
               precision    recall  f1-score   support

           A       0.41      0.44      0.42       394
           B       0.43      0.20      0.28       372
           C       0.51      0.63      0.56       394
           D       0.64      0.73      0.68       454

    accuracy                           0.51      1614
   macro avg       0.50      0.50      0.49      1614
weighted avg       0.50      0.51      0.50      1614



In [10]:
train_inputs.head()

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1
917,Female,No,32,Yes,Artist,9.0,Low,1.0,Cat_6
3398,Male,Yes,72,Yes,Entertainment,NaN,Average,2.0,Cat_6
2045,Female,No,33,Yes,Entertainment,1.0,Low,4.0,Cat_6
8060,Female,Yes,48,Yes,Artist,0.0,Average,6.0,Cat_6
4604,Female,Yes,28,No,Doctor,9.0,Low,1.0,Cat_7


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [11]:
train_inputs.head(5)

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1
917,Female,No,32,Yes,Artist,9.0,Low,1.0,Cat_6
3398,Male,Yes,72,Yes,Entertainment,NaN,Average,2.0,Cat_6
2045,Female,No,33,Yes,Entertainment,1.0,Low,4.0,Cat_6
8060,Female,Yes,48,Yes,Artist,0.0,Average,6.0,Cat_6
4604,Female,Yes,28,No,Doctor,9.0,Low,1.0,Cat_7


In [12]:
# Identify categorical and numeric columns
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes(include='object').columns.tolist()

# Impute missing values
train_inputs[numeric_cols] = SimpleImputer(strategy='mean').fit_transform(train_inputs[numeric_cols])
train_inputs[categorical_cols] = SimpleImputer(strategy='most_frequent').fit_transform(train_inputs[categorical_cols])

# Get indices of categorical columns
cat_feature_indices = [train_inputs.columns.get_loc(col) for col in categorical_cols]

In [13]:
cat_feature_indices

[0, 1, 3, 4, 6, 8]

In [14]:
# Apply SMOTENC
smote_nc = SMOTENC(categorical_features=cat_feature_indices, random_state=42)
X_resampled_smote, y_resampled_smote = smote_nc.fit_resample(train_inputs, train_targets)

In [15]:
train_inputs.shape, X_resampled_smote.shape

((6454, 9), (7256, 9))

In [16]:
train_inputs_transformed = pd.DataFrame(preprocessor.fit_transform(train_inputs), columns=preprocessor.get_feature_names_out())
train_inputs_transformed.head(5)

,num__Age,num__Work_Experience,num__Family_Size,cat__Gender_Female,cat__Gender_Male,cat__Ever_Married_No,cat__Ever_Married_Yes,cat__Graduated_No,cat__Graduated_Yes,cat__Profession_Artist,cat__Profession_Doctor,cat__Profession_Engineer,cat__Profession_Entertainment,cat__Profession_Executive,cat__Profession_Healthcare,cat__Profession_Homemaker,cat__Profession_Lawyer,cat__Profession_Marketing,cat__Var_1_Cat_1,cat__Var_1_Cat_2,cat__Var_1_Cat_3,cat__Var_1_Cat_4,cat__Var_1_Cat_5,cat__Var_1_Cat_6,cat__Var_1_Cat_7,ord__Spending_Score
0,0.197183,0.642857,0.000,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.760563,0.190821,0.125,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,0.211268,0.071429,0.375,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.422535,0.000000,0.625,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0.140845,0.642857,0.000,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [17]:
# Apply SMOTE-Tomek
smote_tomek = SMOTETomek(random_state=42)
X_resampled_tomek, y_resampled_tomek = smote_tomek.fit_resample(train_inputs_transformed, train_targets)

In [18]:
train_inputs_transformed.shape, X_resampled_tomek.shape

((6454, 26), (5666, 26))

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [19]:
test_inputs_transformed = pd.DataFrame(preprocessor.fit_transform(test_inputs), columns=preprocessor.get_feature_names_out())

In [20]:
# One-vs-Rest without resampling
log_reg = LogisticRegression(solver='liblinear')
ovr_model = OneVsRestClassifier(log_reg)
ovr_model.fit(train_inputs_transformed, train_targets)
ovr_predictions = ovr_model.predict(test_inputs_transformed)

print(classification_report(test_targets, ovr_predictions))

              precision    recall  f1-score   support

           A       0.41      0.45      0.43       394
           B       0.41      0.15      0.22       372
           C       0.49      0.65      0.56       394
           D       0.65      0.75      0.70       454

    accuracy                           0.51      1614
   macro avg       0.49      0.50      0.48      1614
weighted avg       0.50      0.51      0.49      1614



In [21]:
# One-vs-Rest with SMOTENC oversampling
X_resampled_smote_transformed = pd.DataFrame(preprocessor.fit_transform(X_resampled_smote), columns=preprocessor.get_feature_names_out())

ovr_model.fit(X_resampled_smote_transformed, y_resampled_smote)
ovr_predictions = ovr_model.predict(test_inputs_transformed)

print(classification_report(test_targets, ovr_predictions))

              precision    recall  f1-score   support

           A       0.42      0.48      0.45       394
           B       0.42      0.22      0.29       372
           C       0.50      0.61      0.55       394
           D       0.67      0.71      0.69       454

    accuracy                           0.52      1614
   macro avg       0.50      0.51      0.49      1614
weighted avg       0.51      0.52      0.50      1614



In [22]:
# One-vs-Rest with Smote-Tomek undersampling
ovr_model.fit(X_resampled_tomek, y_resampled_tomek)
ovr_predictions = ovr_model.predict(test_inputs_transformed)

print(classification_report(test_targets, ovr_predictions))

              precision    recall  f1-score   support

           A       0.41      0.47      0.44       394
           B       0.45      0.21      0.28       372
           C       0.49      0.63      0.55       394
           D       0.68      0.71      0.70       454

    accuracy                           0.52      1614
   macro avg       0.51      0.50      0.49      1614
weighted avg       0.51      0.52      0.50      1614



Найкраща модель — SMOTENC або SMOTE-Tomek, обидві мають незначну, але кращу якість порівняно з базовою моделлю без ресемплінгу.

Клас B має найнижчий F1-score у всіх моделях — навіть після ресемплінгу.

Немає суттєвої різниці між моделями, можливо тому що у даних нечітка межа між класами або ознаки неінформативні.

Також причиною може бути те що в даній ситуації Логістична регресія не підходить і варто спробувати інші моделі